In [2]:
import os
import sys
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
from xgboost import XGBClassifier
from sklearn.svm import SVC
from tqdm.notebook import tqdm


## Function to Perform Classification

In [3]:
def get_classification(classifier, classifier_initialisation, cv_methods, cv_strategies, X, y_encoded, participants, focus, num_class):

    if num_class == 2:
        df_clf_result= pd.DataFrame(columns=['Classifier', 'CV', 'Accuracy', 
                                             'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                             'Best Group', 'Best Accuracy','Best Confusion Matrix' ])
    elif num_class == 3:
        df_clf_result= pd.DataFrame(columns=['Classifier', 'CV', 'Accuracy', 
                                             'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                             'Best Group', 'Best Accuracy','Best Confusion Matrix' ])
    # Loop through classifiers and perform classification
    for classifier, clf in zip(classifier, classifier_initialisation):

        print(f"-----------------------Initializing  {classifier}----------------------- \n")        
        
        # Loop through methods and perform cross-validation
        for cv_method, cv_strategy in zip(cv_methods, cv_strategies):
            
            cv = cv_strategy
            print(f'*****Performing evaluation with {cv_method}*****\n' )
            accuracy_scores = []
            conf_matrices = []
            class_report = {}

            #Store the highest accuracy and confusion matrix to find best group
            max_accuracy = 0
            best_conf_matrix = 0
            best_group = None

            #assign group of focus for cross validation
            group = focus if cv_method in ['Leave-One-Algorithm-Out CV', 'Leave-One-Electrode-Out CV'] else participants
            

            if cv_method == 'Leave-One-Algorithm-Out CV':
                group_name = 'Algorithm'
            elif cv_method == 'Leave-One-Electrode-Out CV':
                group_name = 'Electrode'   
            elif cv_method == 'Leave-One-Subject-Out CV':
                group_name = 'Participant'
            else:
                group_name = 'None' 

            for fold,(train_index, test_index) in enumerate(cv.split(X, y_encoded, groups= group)):
                train_X, test_X = X.iloc[train_index], X.iloc[test_index]
                train_y, test_y = y_encoded[train_index], y_encoded[test_index]
                
                # algorithm chosen for testing in this fold
                test_group = group.iloc[test_index].unique()

                # Initialize and train your Random Forest classifier
                clf.fit(train_X, train_y)

                # Make predictions on the test set
                predictions = clf.predict(test_X)

                # Accuracy
                accuracy = np.round((accuracy_score(test_y, predictions))*100, decimals=4)
                accuracy_scores.append(accuracy)

                # Confusion matrix
                conf_matrix = confusion_matrix(test_y, predictions)
                conf_matrices.append(conf_matrix)
                
                #Classification report 
                classification_rep = classification_report (test_y, predictions, output_dict=True)
                
                #Check for highest accuracy
                if accuracy > max_accuracy:
                    max_accuracy = accuracy
                    best_group = test_group
                    best_conf_matrix = conf_matrix

                # Print accuracy for each fold
                print(f'{fold + 1}. Testing {group_name} : {test_group} , Accuracy: {accuracy}, confusion matrix: {conf_matrix}')

                # Print classification report
                print(f"Classification Report:\n")
                for label, metrics in classification_rep.items():
                    if label != 'accuracy' and label != 'weighted avg' and label != 'macro avg' and label != 'micro avg':

                        if label not in class_report:
                            class_report[label] = {'precision': [], 'recall': [], 'f1-score': [], 'support': []}
                        
                        print(f"Class {label}:")
                        print(f"  Precision: {metrics['precision']:.2f}")
                        print(f"  Recall: {metrics['recall']:.2f}")
                        print(f"  F1 Score: {metrics['f1-score']:.2f}")
                        print(f"  Support: {metrics['support']}")
                        print()

                        class_report[label]['precision'].append(metrics['precision'])
                        class_report[label]['recall'].append(metrics['recall'])
                        class_report[label]['f1-score'].append(metrics['f1-score'])
                        class_report[label]['support'].append(metrics['support'])
                print("\n")
                
            #Average Accuracy over all folds
            average_accuracy = np.round((sum(accuracy_scores) / len(accuracy_scores)), decimals=4)
            print(f"\n Average Accuracy: {average_accuracy}")

            # Pad smaller matrices with zeros before summing to avoid shape mismatch in multiclass classification
            max_shape = max(cm.shape for cm in conf_matrices)
            conf_matrices_padded = [np.pad(cm, ((0, max_shape[0] - cm.shape[0]), (0, max_shape[1] - cm.shape[1])), 'constant') for cm in conf_matrices]

            #Average Confusion Matrix over all folds
            average_conf_matrix = sum(conf_matrices_padded) / len(conf_matrices_padded)
            print(f"\n Average Confusion Matrix: {np.round(average_conf_matrix, decimals=0)} \n")

            #Store the metrics in percentage
            average_class_reports = {}
            for label, metrics in class_report.items():
                average_class_reports[label] = {
                    'precision': np.round((np.mean(metrics['precision'])*100), decimals=2),
                    'recall': np.round((np.mean(metrics['recall'])*100), decimals=2),
                    'f1-score': np.round((np.mean(metrics['f1-score'])*100), decimals=2),
                    'support': np.round((np.mean(metrics['support'])*100), decimals=2)
                }
            # Print the average classification report for each class
            print("Average Classification Report across all folds for each class:")
            for label, metrics in average_class_reports.items():
                print(f"Class {label}:")
                print(f"  Precision: {metrics['precision']:.2f}")
                print(f"  Recall: {metrics['recall']:.2f}")
                print(f"  F1 Score: {metrics['f1-score']:.2f}")
                print(f"  Support: {metrics['support']:.2f}")
                print()

            if cv_method in ['Leave-One-Algorithm-Out CV', 'Leave-One-Electrode-Out CV', 'Leave-One-Subject-Out CV']:
                print(f'\n Maximum Accuracy --- Testing {group_name} : {best_group} , Accuracy: {max_accuracy}, confusion matrix : {best_conf_matrix} \n')
            
            else:
                group_name = None
                best_group = None
                max_accuracy = None
                best_conf_matrix = None
            #Save the result into a dataframe
            if num_class == 2:
                df_clf_result.loc[len(df_clf_result)]= [classifier, cv_method, average_accuracy, 
                                                        average_class_reports['0']['precision'], average_class_reports['0']['recall'], average_class_reports['0']['f1-score'], average_class_reports['0']['support'],
                                                        average_class_reports['1']['precision'], average_class_reports['1']['recall'], average_class_reports['1']['f1-score'], average_class_reports['1']['support'],
                                                        best_group, max_accuracy, best_conf_matrix]
            elif num_class == 3:
                df_clf_result.loc[len(df_clf_result)]= [classifier, cv_method, average_accuracy, 
                                                        average_class_reports['0']['precision'], average_class_reports['0']['recall'], average_class_reports['0']['f1-score'], average_class_reports['0']['support'],
                                                        average_class_reports['1']['precision'], average_class_reports['1']['recall'], average_class_reports['1']['f1-score'], average_class_reports['1']['support'],
                                                        average_class_reports['2']['precision'], average_class_reports['2']['recall'], average_class_reports['2']['f1-score'], average_class_reports['2']['support'],
                                                        best_group, max_accuracy, best_conf_matrix]
        
    return df_clf_result

# Data Processed with Common Average Referencing 

In [3]:
# create folder to store results if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Classification/CAR"
if not os.path.exists(result_path):
    os.makedirs(result_path)

In [4]:
df_processed = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/processed_data.csv").drop("Unnamed: 0", axis=1, errors="ignore")

df_processed

,Focus,Data,FrequencyBand,FilePath
0,Algorithm,Baseline,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
1,Algorithm,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
2,Algorithm,Baseline,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
3,Algorithm,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
4,ElectrodePosition,Baseline,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
5,ElectrodePosition,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
6,ElectrodePosition,Baseline,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
7,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...


## Binary Classification

In [5]:
# Log the outputs

log_file = open(result_path+'/binary_classification_output.log', 'w')
sys.stdout = log_file

In [6]:


# Define the number of classes for classification
num_class = 2
df_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier','CV','Accuracy', 
                                  'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                  'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                  'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
       
    # Binary Classification
    df_brain_waves = df_brain_waves[df_brain_waves['SkillLevel'].isin(['Novice', 'Expert'])]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(random_state=42)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']

        # Add a new row to df_result
        df_result.loc[len(df_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy, 
                                         expert_precision, expert_recall, expert_f1_score, expert_support,
                                         novice_precision, novice_recall, novice_f1_score, novice_support,
                                          best_group, max_accuracy]


df_result.to_csv(result_path+'/bin_clf_result.csv')
df_result
    

  0%|          | 0/8 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,Expert F1-Score,Expert Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",RF,Stratified10-fold CV,85.1370,86.45,87.26,86.78,4080.00,83.76,82.44,82.98,3190.00,None,NaN
1,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Subject-Out CV,72.6083,60.00,53.91,56.64,2040.00,47.83,32.05,35.11,1386.96,[7],100.0
2,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Algorithm-Out CV,85.4017,87.16,87.26,87.02,1275.00,83.71,83.14,83.09,996.88,[BogoSort],96.0
3,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",SVM,Stratified10-fold CV,83.7614,80.76,93.62,86.67,4080.00,89.69,71.15,79.17,3190.00,None,NaN
4,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",SVM,Leave-One-Subject-Out CV,83.7500,75.00,74.61,74.80,2550.00,64.29,64.29,64.29,2278.57,[71],100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",SVM,Leave-One-Subject-Out CV,84.9375,75.00,72.66,73.71,5200.00,68.75,60.06,61.10,4800.00,[71],100.0
68,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",SVM,Leave-One-Electrode-Out CV,86.2500,84.15,90.75,87.28,1300.00,89.16,81.38,85.02,1200.00,[TP8],88.0
69,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",XGB,Stratified10-fold CV,92.0000,90.84,94.23,92.46,8320.00,93.59,89.57,91.47,7680.00,None,NaN
70,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",XGB,Leave-One-Subject-Out CV,85.4375,63.16,59.54,61.24,4378.95,52.17,43.68,45.54,3339.13,[4],100.0


: 

In [31]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

## Multiclass Classification

In [32]:
# Log the outputs

log_file = open(result_path+'/multiclass_classification_output.log', 'w')
sys.stdout = log_file

In [33]:
# Define the number of classes for classification
num_class = 3

df_mcl_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier', 'CV','Accuracy', 
                                      'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                      'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
    
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=4)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                           X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                            X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        intermediate_precision = row['Intermediate Precision']
        intermediate_recall = row['Intermediate Recall']
        intermediate_f1_score = row['Intermediate F1-Score']
        intermediate_support = row['Intermediate Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']
        # Add a new row to df_result
        df_mcl_result.loc[len(df_mcl_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy,  
                                                 expert_precision, expert_recall, expert_f1_score, expert_support,
                                                intermediate_precision, intermediate_recall, intermediate_f1_score, intermediate_support,
                                                novice_precision, novice_recall, novice_f1_score, novice_support,
                                                best_group, max_accuracy]
        
df_mcl_result.to_csv(result_path+'/mcl_clf_result.csv')
df_mcl_result
    

  0%|          | 0/8 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,...,Intermediate Precision,Intermediate Recall,Intermediate F1-Score,Intermediate Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",RF,Stratified10-fold CV,69.3120,70.66,73.78,...,66.10,62.62,64.05,3450.00,71.55,70.84,70.99,3190.00,None,NaN
1,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Subject-Out CV,56.3933,42.86,28.46,...,34.38,17.51,21.16,1078.12,30.56,20.26,23.06,886.11,[49],100.0000
2,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Algorithm-Out CV,68.5948,70.56,71.63,...,66.01,62.48,63.35,1078.12,72.29,71.21,71.10,996.88,[BogoSort],85.7143
3,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",SVM,Stratified10-fold CV,65.0182,56.59,93.62,...,80.59,33.64,47.21,3450.00,79.97,62.43,69.83,3190.00,None,NaN
4,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",SVM,Leave-One-Subject-Out CV,54.0585,44.44,43.63,...,27.78,4.11,6.67,1916.67,47.06,44.01,45.39,1876.47,[71],100.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",SVM,Leave-One-Subject-Out CV,64.2736,54.55,52.49,...,33.33,19.92,22.91,3200.00,45.00,37.27,38.78,3840.00,[71],100.0000
68,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",SVM,Leave-One-Electrode-Out CV,68.6656,64.08,90.02,...,73.50,47.79,57.54,1200.00,73.68,66.41,69.68,1200.00,[FC4],72.9730
69,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",XGB,Stratified10-fold CV,77.9561,73.07,83.17,...,77.89,68.75,72.94,7680.00,84.56,81.50,82.94,7680.00,None,NaN
70,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",XGB,Leave-One-Subject-Out CV,54.2652,52.17,38.38,...,35.48,13.61,16.29,2477.42,34.38,21.97,24.68,2400.00,[24],100.0000


In [34]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

# Data Processed with Common Average Referencing with extracted Channels

In [2]:
# create folder to store results if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Classification/AR_EC"
if not os.path.exists(result_path):
    os.makedirs(result_path)

In [3]:
df_processed = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/processed_data.csv")
df_processed = df_processed.tail(1)
df_processed

,Unnamed: 0,Focus,Data,FrequencyBand,FilePath
7,7,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...


In [4]:
df_channels_extracted = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ2/csp_channels_and_weights.csv")
#Removing 4 bad channels
channel_selected = df_channels_extracted.loc[0:59]["Channel"]
channel_selected

0      FC1
1      PO4
2       Cz
3       P2
4       P4
5       P6
6      FC2
7      POz
8       C1
9      CP4
10      C2
11      Fz
12      F1
13      P7
14      Pz
15     CPz
16    FT10
17     AF3
18      F3
19     CP2
20     FT8
21     CP6
22     FC3
23      P8
24      F2
25     PO3
26      P5
27      F4
28     TP8
29     CP3
30     CP1
31     TP9
32      P1
33     PO9
34     PO7
35      F7
36     CP5
37     PO8
38      P3
39     TP7
40     FC4
41      F8
42      F5
43      C3
44     FC6
45    TP10
46      C4
47      O2
48     AF7
49     Fp1
50      T8
51      O1
52     AF4
53     AF8
54      F6
55      C5
56      Oz
57     FT9
58     FC5
59     FT7
Name: Channel, dtype: object

## Binary Classification

In [5]:
# Log the outputs

log_file = open(result_path+'/binary_classification_output.log', 'w')
sys.stdout = log_file

In [8]:


# Define the number of classes for classification
num_class = 2
df_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier','CV','Accuracy', 
                                  'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                  'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                  'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
       
    # Binary Classification
    df_brain_waves = df_brain_waves[df_brain_waves['SkillLevel'].isin(['Novice', 'Expert'])]

    # Remove least contributing channels in CSP
    df_brain_waves = df_brain_waves[df_brain_waves['Channel'].isin(channel_selected)]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(random_state=42)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']

        # Add a new row to df_result
        df_result.loc[len(df_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy, 
                                         expert_precision, expert_recall, expert_f1_score, expert_support,
                                         novice_precision, novice_recall, novice_f1_score, novice_support,
                                          best_group, max_accuracy]


df_result.to_csv(result_path+'/bin_clf_result.csv')
df_result
    

  0%|          | 0/1 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,Expert F1-Score,Expert Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",RF,Stratified10-fold CV,92.1333,91.29,93.97,92.56,7800.00,93.33,90.14,91.64,7200.00,None,NaN
1,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",RF,Leave-One-Subject-Out CV,84.2000,63.16,60.88,61.96,4105.26,52.38,45.16,47.94,3428.57,[4],100.0
2,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",RF,Leave-One-Electrode-Out CV,92.5333,91.98,94.10,92.91,1300.00,93.66,90.83,92.08,1200.00,[FC3],100.0
3,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",SVM,Stratified10-fold CV,84.4667,81.47,91.03,85.92,7800.00,88.98,77.36,82.64,7200.00,None,NaN
4,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",SVM,Leave-One-Subject-Out CV,84.0000,75.00,73.75,74.34,4875.00,55.56,51.11,51.85,4000.00,[71],100.0
5,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",SVM,Leave-One-Electrode-Out CV,84.6000,81.60,91.03,86.01,1300.00,88.98,77.64,82.85,1200.00,[CP4],88.0
6,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",XGB,Stratified10-fold CV,91.0000,90.94,91.92,91.38,7800.00,91.28,90.00,90.58,7200.00,None,NaN
7,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",XGB,Leave-One-Subject-Out CV,82.4667,63.16,59.12,61.01,4105.26,47.83,40.80,43.58,3130.43,[4],100.0
8,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1500, 4)","(1500,)",XGB,Leave-One-Electrode-Out CV,91.0000,90.74,92.44,91.42,1300.00,91.95,89.44,90.49,1200.00,[TP8],96.0


In [9]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

## Multiclass Classification

In [12]:
# Log the outputs

log_file = open(result_path+'/multi_classification_output.log', 'w')
sys.stdout = log_file

In [13]:
# Define the number of classes for classification
num_class = 3

df_mcl_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier', 'CV','Accuracy', 
                                      'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                      'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
    

    # Remove least contributing channels in CSP
    df_brain_waves = df_brain_waves[df_brain_waves['Channel'].isin(channel_selected)]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=4)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                           X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                            X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        intermediate_precision = row['Intermediate Precision']
        intermediate_recall = row['Intermediate Recall']
        intermediate_f1_score = row['Intermediate F1-Score']
        intermediate_support = row['Intermediate Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']
        # Add a new row to df_result
        df_mcl_result.loc[len(df_mcl_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy,  
                                                 expert_precision, expert_recall, expert_f1_score, expert_support,
                                                intermediate_precision, intermediate_recall, intermediate_f1_score, intermediate_support,
                                                novice_precision, novice_recall, novice_f1_score, novice_support,
                                                best_group, max_accuracy]
        
df_mcl_result.to_csv(result_path+'/mcl_clf_result.csv')
df_mcl_result
    

  0%|          | 0/1 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,...,Intermediate Precision,Intermediate Recall,Intermediate F1-Score,Intermediate Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",RF,Stratified10-fold CV,79.4594,72.77,87.05,...,81.78,68.47,74.45,7200.00,86.82,82.22,84.43,7200.00,None,NaN
1,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",RF,Leave-One-Subject-Out CV,57.9730,52.17,40.58,...,32.26,14.30,16.48,2322.58,40.74,28.46,30.78,2666.67,[24],100.0000
2,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",RF,Leave-One-Electrode-Out CV,79.7748,72.38,87.44,...,83.52,68.61,74.92,1200.00,88.23,82.64,85.10,1200.00,[FC5],89.1892
3,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",SVM,Stratified10-fold CV,68.5586,63.83,89.87,...,73.72,47.08,57.36,7200.00,73.21,66.94,69.81,7200.00,None,NaN
4,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",SVM,Leave-One-Subject-Out CV,64.9550,54.55,52.73,...,31.82,21.82,24.99,3272.73,42.86,36.35,37.36,3428.57,[71],100.0000
5,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",SVM,Leave-One-Electrode-Out CV,68.6487,63.99,90.00,...,73.78,47.22,57.23,1200.00,73.55,66.94,69.95,1200.00,[Fz],75.6757
6,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",XGB,Stratified10-fold CV,77.8378,72.63,81.67,...,77.13,69.17,72.88,7200.00,85.34,82.36,83.76,7200.00,None,NaN
7,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",XGB,Leave-One-Subject-Out CV,56.7117,54.55,41.14,...,34.38,13.75,16.39,2250.00,34.38,23.54,26.09,2250.00,[24],100.0000
8,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2220, 4)","(2220,)",XGB,Leave-One-Electrode-Out CV,77.1622,72.08,81.41,...,77.95,68.33,72.11,1200.00,84.92,81.39,82.81,1200.00,[FC3],89.1892


In [ ]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

# Data Processed with Baseline Correction Algorithm

In [37]:

# create folder to store results if not exist
result_path_bca= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Classification/BCA"
if not os.path.exists(result_path_bca):
    os.makedirs(result_path_bca)

df_processed_bca = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/BCA/processed_data.csv").drop("Unnamed: 0", axis=1, errors="ignore")
df_processed = df_processed_bca.copy()
df_processed

,Focus,Data,FrequencyBand,FilePath
0,Algorithm,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
1,Algorithm,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
2,ElectrodePosition,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
3,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...


## Binary classification

In [41]:
# Log the outputs

log_file = open(result_path_bca+'/binary_classification_output.log', 'a')
sys.stdout = log_file

In [42]:


# Define the number of classes for classification
num_class = 2
df_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier','CV','Accuracy', 
                                  'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                  'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                  'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
       
    # Binary Classification
    df_brain_waves = df_brain_waves[df_brain_waves['SkillLevel'].isin(['Novice', 'Expert'])]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(random_state=42)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']

        # Add a new row to df_result
        df_result.loc[len(df_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy, 
                                         expert_precision, expert_recall, expert_f1_score, expert_support,
                                         novice_precision, novice_recall, novice_f1_score, novice_support,
                                          best_group, max_accuracy]


df_result.to_csv(result_path_bca+'/bin_clf_result.csv')
df_result
    

  0%|          | 0/4 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score a

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,Expert F1-Score,Expert Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Stratified10-fold CV,84.8744,87.03,86.02,86.45,4080.00,82.55,83.37,82.84,3190.00,None,NaN
1,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Subject-Out CV,76.4667,60.00,53.12,56.20,2040.00,40.00,33.97,36.32,1276.00,[55],100.0000
2,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Algorithm-Out CV,85.2868,87.08,87.28,86.87,1275.00,84.31,82.51,82.82,996.88,[SiebDesEratosthenes],95.2381
3,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Stratified10-fold CV,83.7595,80.76,93.62,86.67,4080.00,89.64,71.15,79.16,3190.00,None,NaN
4,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Leave-One-Subject-Out CV,83.7500,75.00,74.61,74.80,2550.00,60.00,60.00,60.00,2126.67,[71],100.0000
5,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Leave-One-Algorithm-Out CV,83.7415,80.62,93.73,86.64,1275.00,89.53,70.68,78.88,996.88,[HeightOfTree],87.5000
6,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",XGB,Stratified10-fold CV,89.5491,88.95,92.90,90.86,4080.00,90.58,85.26,87.79,3190.00,None,NaN
7,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",XGB,Leave-One-Subject-Out CV,81.7917,60.00,57.03,58.31,2040.00,55.56,50.23,52.54,1772.22,[4],100.0000
8,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",XGB,Leave-One-Algorithm-Out CV,89.2997,89.40,91.99,90.54,1275.00,89.64,85.88,87.49,996.88,[SmallGauss],96.0000
9,Algorithm,CodeComprehension,AlphaBetaThetaGamma,"(727, 4)","(727,)",RF,Stratified10-fold CV,92.9776,92.02,95.83,93.88,4080.00,94.35,89.32,91.76,3190.00,None,NaN


In [15]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

## Multi-class Classification

In [38]:
# Log the outputs

log_file = open(result_path_bca+'/multiclass_classification_output.log', 'a')
sys.stdout = log_file

In [39]:
# Define the number of classes for classification
num_class = 3

df_mcl_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier', 'CV','Accuracy', 
                                      'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                      'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
    
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=4)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                           X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                            X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        intermediate_precision = row['Intermediate Precision']
        intermediate_recall = row['Intermediate Recall']
        intermediate_f1_score = row['Intermediate F1-Score']
        intermediate_support = row['Intermediate Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']
        # Add a new row to df_result
        df_mcl_result.loc[len(df_mcl_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy,  
                                                 expert_precision, expert_recall, expert_f1_score, expert_support,
                                                intermediate_precision, intermediate_recall, intermediate_f1_score, intermediate_support,
                                                novice_precision, novice_recall, novice_f1_score, novice_support,
                                                best_group, max_accuracy]
        
df_mcl_result.to_csv(result_path_bca+'/mcl_clf_result.csv')
df_mcl_result
    

  0%|          | 0/4 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score a

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,...,Intermediate Precision,Intermediate Recall,Intermediate F1-Score,Intermediate Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",RF,Stratified10-fold CV,65.9545,69.04,68.12,...,58.81,58.59,58.50,3450.00,70.38,71.18,70.38,3190.00,None,NaN
1,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Subject-Out CV,56.2962,48.00,32.50,...,31.25,16.19,19.19,1078.12,28.57,21.50,23.87,911.43,[6],100.0000
2,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Algorithm-Out CV,66.4041,69.45,69.59,...,59.68,58.69,58.74,1078.12,71.94,70.32,70.38,996.88,[GreatestCommonDivisor],83.8710
3,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",SVM,Stratified10-fold CV,67.5398,59.12,93.38,...,82.94,41.75,55.22,3450.00,79.19,62.43,69.58,3190.00,None,NaN
4,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",SVM,Leave-One-Subject-Out CV,61.7731,50.00,49.35,...,33.33,20.08,23.91,2300.00,44.44,44.44,44.44,1772.22,[71],100.0000
5,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",SVM,Leave-One-Algorithm-Out CV,68.9938,60.83,93.25,...,82.56,46.62,59.28,1078.12,79.26,61.87,69.27,996.88,[Power],72.7273
6,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",XGB,Stratified10-fold CV,72.6705,71.23,79.89,...,67.51,62.11,64.35,3450.00,81.51,74.94,77.77,3190.00,None,NaN
7,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",XGB,Leave-One-Subject-Out CV,58.4511,52.17,40.62,...,29.03,16.37,18.43,1112.90,40.91,32.77,35.58,1450.00,[6],100.0000
8,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",XGB,Leave-One-Algorithm-Out CV,73.4887,71.03,83.81,...,73.14,58.29,63.86,1078.12,80.41,76.50,78.00,996.88,[Palindrome],85.7143
9,Algorithm,CodeComprehension,AlphaBetaThetaGamma,"(1072, 4)","(1072,)",RF,Stratified10-fold CV,79.7577,75.98,87.99,...,77.00,70.44,73.32,3450.00,90.08,79.33,84.13,3190.00,None,NaN


In [40]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

# Data Processed with Baseline Correction Algorithm with extracted Channels

In [21]:
# create folder to store results if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Classification/BCA_EC"
if not os.path.exists(result_path):
    os.makedirs(result_path)


In [22]:

df_processed = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/BCA/processed_data.csv")
df_processed = df_processed.tail(1)
df_processed


,Unnamed: 0,Focus,Data,FrequencyBand,FilePath
3,3,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...


In [23]:
df_channels_extracted = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ2/BCA/csp_channels_and_weights.csv")
#Removing 4 bad channels
channel_selected = df_channels_extracted.loc[0:59]["Channel"]
channel_selected




0      PO3
1       P1
2       P6
3       P2
4       P4
5      POz
6      PO4
7       P3
8       P5
9      PO7
10      Pz
11     TP7
12     TP9
13    TP10
14     CP6
15      P8
16      P7
17     TP8
18    FT10
19     FC6
20      C6
21     CP4
22      Oz
23     CP2
24      O1
25     PO8
26     FT9
27     CP3
28      O2
29     PO9
30     FT8
31     CP5
32      C4
33     CPz
34     CP1
35    PO10
36      C5
37     FT7
38      T8
39      F8
40     FC4
41      T7
42      C3
43      C2
44     FC3
45     Fp2
46      F6
47     FC5
48     AF4
49      C1
50      Cz
51     AF3
52     AF8
53     AF7
54      F7
55     Fp1
56      F5
57      F4
58      F3
59     FC2
Name: Channel, dtype: object

## Binary Classification

In [24]:
# Log the outputs
log_file = open(result_path+'/binary_classification_output.log', 'w')
sys.stdout = log_file


In [25]:


# Define the number of classes for classification
num_class = 2
df_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier','CV','Accuracy', 
                                  'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                  'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                  'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
       
    # Binary Classification
    df_brain_waves = df_brain_waves[df_brain_waves['SkillLevel'].isin(['Novice', 'Expert'])]

    # Remove least contributing channels in CSP
    df_brain_waves = df_brain_waves[df_brain_waves['Channel'].isin(channel_selected)]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(random_state=42)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']

        # Add a new row to df_result
        df_result.loc[len(df_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy, 
                                         expert_precision, expert_recall, expert_f1_score, expert_support,
                                         novice_precision, novice_recall, novice_f1_score, novice_support,
                                          best_group, max_accuracy]


df_result.to_csv(result_path+'/bin_clf_result.csv')
df_result
    
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__


  0%|          | 0/1 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

  0%|          | 0/1 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

## Multiclass Classification

In [ ]:

# Log the outputs

log_file = open(result_path+'/multi_classification_output.log', 'w')
sys.stdout = log_file
# Define the number of classes for classification
num_class = 3

df_mcl_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier', 'CV','Accuracy', 
                                      'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                      'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
    

    # Remove least contributing channels in CSP
    df_brain_waves = df_brain_waves[df_brain_waves['Channel'].isin(channel_selected)]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=4)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                           X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                            X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        intermediate_precision = row['Intermediate Precision']
        intermediate_recall = row['Intermediate Recall']
        intermediate_f1_score = row['Intermediate F1-Score']
        intermediate_support = row['Intermediate Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']
        # Add a new row to df_result
        df_mcl_result.loc[len(df_mcl_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy,  
                                                 expert_precision, expert_recall, expert_f1_score, expert_support,
                                                intermediate_precision, intermediate_recall, intermediate_f1_score, intermediate_support,
                                                novice_precision, novice_recall, novice_f1_score, novice_support,
                                                best_group, max_accuracy]
        
df_mcl_result.to_csv(result_path+'/mcl_clf_result.csv')
df_mcl_result
    
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

# Data Processed with Common Average Referencing Algorithm + Channels

In [4]:
# create folder to store results if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Classification/Algorithm_and_Channel"
if not os.path.exists(result_path):
    os.makedirs(result_path)

In [5]:
df_processed = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/P3/features_ccdata.csv").drop("Unnamed: 0", axis=1, errors="ignore")

df_processed

,Participant,SkillScore,SkillLevel,Algorithm,Channel,Alpha_Mean,Beta_Mean,Gamma_Mean,Theta_Mean,Alpha_StdDev,...,Gamma_Median,Theta_Median,Alpha_ZeroCrossRate,Beta_ZeroCrossRate,Gamma_ZeroCrossRate,Theta_ZeroCrossRate,Alpha_Entropy,Beta_Entropy,Gamma_Entropy,Theta_Entropy
0,1,0.331385,Intermediate,IsPrime,Fp1,0.166390,0.150510,0.090489,0.157715,0.047104,...,0.086674,0.166295,0.290323,0.354839,0.322581,0.354839,3.423196,3.451393,3.450221,3.421571
1,1,0.331385,Intermediate,IsPrime,Fp2,0.162938,0.180551,0.085663,0.183140,0.047777,...,0.087033,0.173035,0.322581,0.322581,0.354839,0.290323,3.422616,3.453243,3.444997,3.435099
2,1,0.331385,Intermediate,IsPrime,F7,0.137152,0.190865,0.095615,0.166988,0.047963,...,0.093597,0.154126,0.290323,0.354839,0.322581,0.387097,3.403825,3.448273,3.436679,3.412071
3,1,0.331385,Intermediate,IsPrime,F3,0.171606,0.141907,0.056353,0.227676,0.053040,...,0.057599,0.206876,0.290323,0.322581,0.290323,0.354839,3.418012,3.434288,3.439739,3.415109
4,1,0.331385,Intermediate,IsPrime,Fz,0.184896,0.130419,0.052744,0.244464,0.065693,...,0.050492,0.230446,0.322581,0.354839,0.322581,0.354839,3.404437,3.441623,3.419171,3.418192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68603,71,0.435651,Expert,IsPrime,PO7,0.216094,0.181458,0.039566,0.231026,0.062641,...,0.035469,0.231366,0.290323,0.322581,0.322581,0.290323,3.423160,3.444693,3.405300,3.414672
68604,71,0.435651,Expert,IsPrime,PO3,0.222476,0.188065,0.039012,0.203466,0.079452,...,0.037877,0.203373,0.290323,0.322581,0.290323,0.354839,3.407981,3.445670,3.427560,3.413501
68605,71,0.435651,Expert,IsPrime,POz,0.214923,0.176492,0.039372,0.211699,0.073116,...,0.037066,0.210536,0.354839,0.290323,0.322581,0.387097,3.413788,3.449976,3.426663,3.416950
68606,71,0.435651,Expert,IsPrime,PO4,0.164153,0.175065,0.050551,0.213906,0.052657,...,0.049701,0.205694,0.387097,0.387097,0.258065,0.322581,3.418935,3.450560,3.443043,3.403913


## Binary Classification

In [6]:
# Log the outputs
log_file = open(result_path+'/binary_classification_output.log', 'w')
sys.stdout = log_file


In [9]:


# Define the number of classes for classification
num_class = 2
df_result = pd.DataFrame(columns=['X_shape','y_shape','Classifier','CV','Accuracy', 
                                  'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                  'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                  'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    
    df_brain_waves = df_processed.copy()
       
    # Binary Classification
    df_brain_waves = df_brain_waves[df_brain_waves['SkillLevel'].isin(['Novice', 'Expert'])]
    
   
    X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(random_state=42)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    algorithms = df_brain_waves["Algorithm"]        
    cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
    cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                            GroupKFold(n_splits=len(participants.unique())),
                            GroupKFold(n_splits=len(algorithms.unique()))]
    cv_methods = cv_methods_algorithm
    cv_strategies = cv_strategies_algorithm
    df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, algorithms, num_class)
    


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']

        # Add a new row to df_result
        df_result.loc[len(df_result)] = [X_shape, y_shape, classifier, cv, accuracy, 
                                         expert_precision, expert_recall, expert_f1_score, expert_support,
                                         novice_precision, novice_recall, novice_f1_score, novice_support,
                                          best_group, max_accuracy]


df_result.to_csv(result_path+'/bin_clf_result.csv')
df_result
    

  0%|          | 0/68608 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

KeyboardInterrupt: 